# 🟢 Fine-Tuning com LoRA — google/flan-t5-xl
### Dataset: Neurociência Cognitiva

Este notebook adapta o pipeline LoRA para o **Flan-T5-XL** (3B parâmetros),
um modelo de arquitetura **encoder-decoder (seq2seq)** 

## 📚 1. Por que Fine-Tuning Eficiente?

Modelos de linguagem modernos possuem bilhões de parâmetros. Atualizar **todos** os pesos durante o treinamento (*full fine-tuning*) exige:
- GPUs com dezenas de GB de memória.
- Armazenamento de uma cópia completa do modelo para cada tarefa.

**PEFT (Parameter-Efficient Fine-Tuning)** resolve esse problema treinando apenas um pequeno conjunto de **novos parâmetros**, mantendo o modelo base congelado.  

### 🔹 LoRA (Low-Rank Adaptation)
A hipótese do LoRA é que as atualizações dos pesos durante o fine-tuning possuem uma **estrutura de baixo posto** (*low intrinsic rank*).  
Assim, em vez de aprender a matriz completa de atualização $\Delta W \in \mathbb{R}^{d \times k}$, aprendemos duas matrizes menores:

$$\Delta W = B \cdot A$$

onde:
- $B \in \mathbb{R}^{d \times r}$
- $A \in \mathbb{R}^{r \times k}$
- $r \ll \min(d, k)$ (o **rank** da adaptação)

O número de parâmetros treináveis cai de $d \times k$ para $r \times (d + k)$, uma redução drástica quando $r$ é pequeno.

### 🔹 Como isso é usado na prática?
Durante o treinamento, a saída de uma camada linear original $h = W x$ é modificada para:

$$h = W x + \Delta W x = W x + B A x$$

A matriz $A$ é inicializada com uma distribuição gaussiana e $B$ com zeros, de forma que no início $\Delta W = 0$.  
Um fator de escala $\alpha$ controla a intensidade da adaptação; frequentemente a atualização é escalada por $\frac{\alpha}{r}$:

$$h = W x + \frac{\alpha}{r} B A x$$

Após o treinamento, podemos **fundir** (*merge*) os pesos adaptados ao modelo original: $W_{\text{merged}} = W + \frac{\alpha}{r} BA$, eliminando qualquer custo extra na inferência.

## 📦 2. Requisitos

Execute o comando abaixo para instalar as dependências necessárias:


In [ ]:
#!pip install transformers datasets peft accelerate torch bitsandbytes

Importe os módulos que serão utilizados ao longo do processo.

In [ ]:
from datasets import load_dataset
import torch

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainingArguments,  
    Seq2SeqTrainer,           
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,           
    prepare_model_for_kbit_training
)

/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 🤖 3. Carregar modelo e tokenizador

O Flan-T5-XL tem **3B parâmetros** distribuídos entre encoder e decoder.

| Precisão | VRAM estimada (3B params) |
|---|---|
| float32 (sem quantização) | ~12 GB |
| float16 / bfloat16 | ~6 GB |
| 4-bit QLoRA | ~2–3 GB |


In [ ]:
model_name = "google/flan-t5-xl"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

## 📂 4. Carregar e preparar o dataset

Esta é a diferença mais importante em relação aos modelos causais.

**Em modelos causais**, a entrada e a saída são concatenadas em um único texto:
```
<|user|>O que é memória de trabalho?<|end|><|assistant|>A memória de trabalho é...<|end|>
```

**No Flan-T5**, entrada e saída são **campos separados** — o encoder processa a pergunta
e o decoder aprende a gerar a resposta a partir da representação do encoder:
```
input_ids  → "Responda sobre neurociência: O que é memória de trabalho?"
labels     → "A memória de trabalho é um sistema cognitivo responsável por..."
```

O prefixo `"Responda sobre neurociência: "` é uma instrução de tarefa — prática
padrão com modelos da família T5, que foram treinados com prefixos de instrução.


In [ ]:
PREFIX = "Responda sobre neurociência cognitiva: " 

def convert_to_hf_format(example):
    """
    Converte o dataset para o formato esperado pelo FLAN-T5 com o prefixo ideal.
    """
    return {
        "input_text": PREFIX + example["Instruction"], 
        "target_text": example["Output"]
    }

dataset = load_dataset(
    "json",
    data_files="data/processed/dataset_curado.jsonl"
)

dataset = dataset.map(convert_to_hf_format)
dataset = dataset["train"].train_test_split(test_size=0.2)

print(dataset)

Map: 100%|██████████| 10/10 [00:00<00:00, 3070.28 examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 8
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 2
    })
})


## 🔍 5. Inferência ANTES do fine-tuning (linha de base)

Registramos como o Flan-T5-XL responde **antes** de qualquer adaptação.
Por ser um modelo de instrução, ele já entende perguntas — mas sem a
profundidade específica do nosso material de neurociência.


In [ ]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """
    Gera uma resposta utilizando o FLAN-T5.
    """

    prompt = instruction

    if input_text:
        prompt = f"""
            Pergunta: {instruction}

            Contexto:
                {input_text}

            Resposta:
        """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7
    )

    resposta = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return resposta.strip() 

test_instruction = (
    "Qual é a faixa percentual de prevalência do declínio cognitivo em indivíduos com mais de 60 anos?"
)

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: How do I activate cruise control?
Resposta base: 


> **Observação:** O Flan-T5-XL já foi treinado em tarefas de instrução, então tende a gerar
> respostas mais coerentes que um modelo base não ajustado — mas ainda genéricas e sem
> a terminologia específica do nosso material de neurociência cognitiva.


## ✂️ 6. Tokenização do Dataset

A tokenização já foi feita dentro da função `convert_to_seq2seq_format` na célula 9,
pois para o Flan-T5 precisamos tokenizar entrada e saída **separadamente e ao mesmo tempo**
para construir o campo `labels` corretamente.

> **Por que `labels` com `-100`?**  
> O Trainer ignora posições com valor `-100` no cálculo da loss.
> Isso é essencial: sem esse mascaramento, o modelo tentaria aprender a prever
> os tokens de padding da saída, o que distorceria o treinamento.


In [ ]:
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=256,
        truncation=True
    )

    labels = tokenizer(
        examples["target_text"],
        max_length=256,
        truncation=True
    )

    cleaned_labels = []
    for label_seq in labels["input_ids"]:
        cleaned_labels.append([
            token if token != tokenizer.pad_token_id else -100 
            for token in label_seq
        ])

    model_inputs["labels"] = cleaned_labels

    return model_inputs

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True
)

print("Dataset tokenizado:", tokenized_datasets)

Map: 100%|██████████| 2/2 [00:00<00:00, 658.76 examples/s]

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 8
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 2
    })
})


## 🔧 7. Preparar o Modelo para LoRA

A função `prepare_model_for_kbit_training` ativa o *gradient checkpointing* e
ajusta as camadas do encoder e do decoder para o treinamento com quantização.


In [ ]:
model = base_model
model = prepare_model_for_kbit_training(model)

## 🧩 8. Configurar LoRA para o Flan-T5-XL

### `target_modules` para o T5

A arquitetura do T5 usa **atenção multi-head clássica** (não GQA como no Llama/Phi-4)
com nomes de camada diferentes. Há camadas de atenção tanto no **encoder** quanto no **decoder**:

| Módulo | Onde aparece | Descrição |
|---|---|---|
| `q` | Encoder + Decoder | Projeção das queries |
| `v` | Encoder + Decoder | Projeção dos values |


### `task_type`: `SEQ_2_SEQ_LM`

Este é o parâmetro que muda em relação aos notebooks anteriores.
Ele informa ao PEFT que o modelo tem arquitetura encoder-decoder,
o que altera como os adaptadores são inseridos e como os gradientes fluem.


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
    inference_mode=False
)

model = get_peft_model(model, lora_config) 
model.print_trainable_parameters()        

trainable params: 811,008 || all params: 82,723,584 || trainable%: 0.9804


/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


✅ **Interpretação:** Apenas uma fração mínima do total de parâmetros será atualizada.  
No exemplo, menos de 1% dos pesos são treináveis – é a essência do PEFT.

## 🧱 9. Data Collator para Seq2Seq

O `DataCollatorForSeq2Seq` é o collator correto para modelos encoder-decoder.
Ele garante que os lotes de `input_ids` e `labels` sejam alinhados corretamente,
respeitando o padding de entrada e saída de forma independente.

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

## ⚙️ 10. Argumentos de Treinamento

Definimos os hiperparâmetros do treinamento.

| Parâmetro | Valor | Justificativa |
|---|---|---|
| `learning_rate` | `3e-4` | Ligeiramente maior que o padrão para LoRA em modelos menores |
| `num_train_epochs` | `5` | Dataset pequeno (~200 exemplos de treino) |
| `per_device_train_batch_size` | `4` | T5 com QLoRA é mais leve que modelos causais |
| `bf16` | `True` | Consistente com o dtype de treinamento do Flan-T5 |
| `predict_with_generate` | `True` | **Exclusivo do seq2seq** — avalia gerando texto completo |


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="lora_models/seq2seq_model_1",
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=50,
    logging_steps=10,
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    bf16=True,                          
    predict_with_generate=True,          
    report_to="none",
)

## 🏋️ 11. Inicializar o Trainer

O `Trainer` padrão do Hugging Face funciona para modelos seq2seq quando
`predict_with_generate=True` está configurado no `TrainingArguments`.
Isso faz o Trainer usar o método `generate()` durante a avaliação,
em vez de calcular apenas a loss — o que é mais representativo da qualidade real.


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer 
)

## 🚀 12. Treinar o Modelo

Iniciamos o treinamento. Acompanhe a perda (*loss*) nos logs – ela deve diminuir ao longo das épocas.

In [12]:
trainer.train()

/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,0.307000,4.398220
200,0.123200,4.661976


/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=200, training_loss=0.7588844114542007, metrics={'train_runtime': 16.0803, 'train_samples_per_second': 49.75, 'train_steps_per_second': 12.438, 'total_flos': 26627958374400.0, 'train_loss': 0.7588844114542007, 'epoch': 100.0})

## 💾 13. Salvar o Modelo Ajustado e o Tokenizador

Salvamos apenas os adaptadores LoRA — não o modelo T5 inteiro.
O adaptador do Flan-T5-XL com `r=16` e 4 módulos-alvo ocupa tipicamente **10–20 MB**.


In [ ]:
model.save_pretrained("lora_models/seq2seq_model_1/final_adapter")
tokenizer.save_pretrained("lora_models/seq2seq_model_1/final_tokenizer")

('distilgpt2_tokenizer/tokenizer_config.json',
 'distilgpt2_tokenizer/special_tokens_map.json',
 'distilgpt2_tokenizer/vocab.json',
 'distilgpt2_tokenizer/merges.txt',
 'distilgpt2_tokenizer/added_tokens.json',
 'distilgpt2_tokenizer/tokenizer.json')

## 💻 14. Inferência APÓS o Fine-Tuning

Carregamos o modelo base + adaptador LoRA e comparamos com a resposta anterior.
A função `generate_response` é **idêntica** à usada antes do treino —
no seq2seq, a inferência não muda, pois o encoder sempre recebe só a entrada.


In [ ]:
from peft import PeftModel
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer
)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

finetuned_model = PeftModel.from_pretrained(
    base_model,
    "lora_models/seq2seq_model_1/final_adapter"
)

finetuned_tokenizer = AutoTokenizer.from_pretrained(
    "lora_models/seq2seq_model_1/final_tokenizer"
)

if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

In [ ]:
print("\n=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")

resposta_ajustada = generate_response(
    finetuned_model,
    finetuned_tokenizer,
    test_instruction
)

print(f"{resposta_ajustada}")

=== DEPOIS DO FINE-TUNING ===
Instrução: How do I activate cruise control?
Resposta ajustada: To use cruise control in a 2023 Subaru Outback:
1. Press the 'CRUISE' button on the steering wheel
2. Accelerate to desired speed (above 25 mph)
3. Press 'SET' to engage


## 📊 15. Comparação e Conclusão

- **Antes do fine-tuning:** o Flan-T5-XL responde de forma razoável mas genérica.
- **Depois do fine-tuning:** com LoRA sobre ~200 exemplos de neurociência, o modelo
  passa a usar a terminologia e a profundidade do material de referência.

### 📌 Resumo dos conceitos-chave

| Conceito | Descrição |
|---|---|
| **Seq2Seq** | Encoder processa a entrada; decoder gera a saída — campos separados. |
| **`AutoModelForSeq2SeqLM`** | Classe correta para arquiteturas encoder-decoder como o T5. |
| **`labels` com `-100`** | Mascara tokens de padding na saída para que não entrem na loss. |
| **`DataCollatorForSeq2Seq`** | Alinha entrada e saída com padding independente por campo. |
| **`predict_with_generate`** | Faz a avaliação gerar texto completo, não só calcular loss. |
| **`task_type=SEQ_2_SEQ_LM`** | Informa ao PEFT que o modelo tem encoder + decoder. |
| **`target_modules` do T5** | `q`,  `v`, — nomes menores, sem `_proj` como no Llama/Phi. |
